In [2]:
import pandas as pd
import os

# === Charger ton CSV ===
data = pd.read_csv('dataset_pitt_cookie.csv')
data = data[data['dx_label'].isin(['ProbableAD', 'Control'])].reset_index(drop=True)


# === Lit le fichier .cha COMPLET ===
def read_full_cha(file_path):
    """Retourne le contenu complet du fichier .cha sans aucun filtrage."""
    if not os.path.exists(file_path):
        return f"[Fichier introuvable : {file_path}]"
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


# === Affichage complet d'un exemple ===
def show_example(row, max_chars=None):
    print("=" * 100)
    print(f"📄 SUJET : {row['subject_id']:<5} | VISITE : {row['visit']} | DIAGNOSTIC : {row['dx_label']}")
    print(f"   MMSE : {row.get('mmse_visit', 'N/A')} | Âge : {row.get('entryage', 'N/A')} | Sex : {row.get('sex', 'N/A')} | Éduc : {row.get('educ', 'N/A')}")
    print(f"   FICHIER : {row['file_path']}")
    print("=" * 100)
    
    # --- 1. FICHIER .CHA COMPLET ---
    raw = read_full_cha(row['file_path'])
    print("\n📝 FICHIER .CHA COMPLET (toutes les lignes : @, *, %) :")
    print("-" * 100)
    if max_chars and len(raw) > max_chars:
        print(raw[:max_chars] + f"\n[...tronqué après {max_chars} caractères, total = {len(raw)}]")
    else:
        print(raw)
    
    # --- 2. TEXTE NETTOYÉ ---
    print("\n\n🧹 TEXTE NETTOYÉ (utilisé par RoBERTa) :")
    print("-" * 100)
    cleaned = row['transcript']
    print(cleaned)
    
    # --- 3. FEATURES CALCULÉES ---
    print("\n\n📊 FEATURES CLINIQUES (comptées sur le texte brut) :")
    print("-" * 100)
    features_info = [
        ('n_filled_pauses',  "Hésitations (&-um, &-uh, &-er)",          "Marqueurs de recherche lexicale"),
        ('n_phon_fragments', "Fragments phonologiques (&+w, &+co)",     "Mots commencés puis abandonnés (signal AD fort)"),
        ('n_paralinguistic', "Paralinguistique (&=laughs, &=sighs)",    "Rires, soupirs, hochements"),
        ('n_retracings',     "Retracings ([/] [//] [///])",             "Répétitions et révisions (signal AD fort)"),
        ('n_unintelligible', "Mots indistincts (xxx, yyy, www)",        "Patient inaudible/incompréhensible"),
        ('n_pauses',         "Pauses silencieuses ((.) (..) (...))",    "Silences mesurés"),
        ('n_tokens',         "Nombre de tokens",                         "Longueur du discours"),
        ('n_utterances',     "Nombre d'énoncés (lignes *PAR)",          "Découpage en phrases"),
    ]
    print(f"{'Feature':<22} {'Valeur':<8} {'Description':<45} {'Sens clinique'}")
    print("-" * 100)
    for col, desc, clinical in features_info:
        if col in row.index:
            value = row[col]
            print(f"{col:<22} {value:<8} {desc:<45} {clinical}")
    
    print("\n" + "=" * 100 + "\n\n")


# === ÉCHANTILLONNAGE : 1 control + 2 AD ===
control_example = data[data['dx_label']=='Control'].sort_values('mmse_visit', ascending=False).iloc[0]
ad_severe = data[data['dx_label']=='ProbableAD'].sort_values('mmse_visit', ascending=True).iloc[0]

ad_moderate_df = data[(data['dx_label']=='ProbableAD') & data['mmse_visit'].between(18, 22)]
ad_moderate = ad_moderate_df.iloc[0] if len(ad_moderate_df) > 0 else None


print("\n" + "🟢"*50)
print("EXEMPLE 1 : CONTROL (sujet sain, MMSE le plus haut)")
print("🟢"*50 + "\n")
show_example(control_example)

if ad_moderate is not None:
    print("\n" + "🟡"*50)
    print("EXEMPLE 2 : AD MODÉRÉ (MMSE entre 18 et 22)")
    print("🟡"*50 + "\n")
    show_example(ad_moderate)

print("\n" + "🔴"*50)
print("EXEMPLE 3 : AD SÉVÈRE (MMSE le plus bas)")
print("🔴"*50 + "\n")
show_example(ad_severe)


🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
EXEMPLE 1 : CONTROL (sujet sain, MMSE le plus haut)
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

📄 SUJET : 2     | VISITE : 0 | DIAGNOSTIC : Control
   MMSE : 30.0 | Âge : 58 | Sex : 0 | Éduc : 16
   FICHIER : Pitt\Control\cookie\002-0.cha

📝 FICHIER .CHA COMPLET (toutes les lignes : @, *, %) :
----------------------------------------------------------------------------------------------------
@UTF8
@PID:	11312/a-00090633-0
@Begin
@Languages:	eng
@Participants:	PAR Participant, INV Investigator
@ID:	eng|Pitt|PAR|58;|female|Control||Participant|30||
@ID:	eng|Pitt|INV|||||Investigator|||
@Media:	002-0, audio
@G:	Cookie
*PAR:	the scene is <in the> [/] in the kitchen . 3754_5640
%mor:	det|the-Def-Art noun|scene aux|be-Fin-Ind-Pres-S3 adp|in det|the-Def-Art noun|kitchen .
%gra:	1|2|DET 2|6|NSUBJ 3|6|COP 4|6|CASE 5|6|DET 6|0|ROOT 7|6|PUNCT
*PAR:	the mother is wiping dishes and the water is running on the floor . 5776_11843
%mor:	